# The forward pass

Up until now, we have focused on essential PyTorch syntax, the tools you need to work with tensors, but not yet the deeper architectural ideas required to actually build a model. Remember the overview lecture? In that lecture, we walked through the five steps of deep learning.

Before we continue, let’s refresh your memory with a short exercise.

## Exercise 7 - The five steps of deep learning 

Can you remember the five steps of deep learning?
Write down everything you remember about these steps, from the holistic idea behind each step to the actual math involved.

When you’re done, compare your notes with the overview lecture and see how much you recall.

**Note**
This exercise has no answer in the ExerciseAnswers notebook. You should compare to the overview lecture. 

 ---
## The model's first guess

The forward pass is when the model runs and produces its predictions. The very first time you feed data through the model, the weights and biases are still random because no learning has happened yet. Since the model knows nothing at this point, its first predictions
will also be essentially random.

Even though this first guess is random, it is extremely important. It becomes the starting point for backpropagation and the adjustments of the weights and biases. **This lecture will show how to implement a model’s first guess using only the tensor operations you have learned so far.** In later lectures, you will learn how to use higher‑level libraries for this, but to truly understand the deep‑learning process, we will first build it ourselves.

To do this, we will create a very simple model: linear regression, a structure you will see everywhere when learning about deep neural networks.

The model is defined as:

- y_hat = X @ W +b

Where
- y_hat = our model's prediction
- X = our input data
- W = the weight
- b = the bias 

W and b are the parameters the model can adjust in order to reduce the loss and improve the accuracy of its predictions.

**The entire goal of deep learning is to find the best values for W and b so that the prediction y_hat becomes as close as possible to the true value y (the ground truth).**

 ---
## Exercise 8 - The @ operator

Do you remember what the "@" operator does? Describe it with 2 sentences. 

 ---
## Create data for this lecture 

In [46]:
import torch

# Our batch have 10 samples 
N = 10
# Each sample has 1 input feature and 1 output value 
D_in = 1
D_out = 1

# Create input data
X = torch.randn(N, D_in)

# Create our "ground truth", i.e. the weight and bias
# that will result in a y_hat with the same value as
# our actual y
true_W = torch.tensor([[2.0]]) # true W is 2.0
true_b = torch.tensor(1.0) # true b is 1.0

##!! In real-world applications the model never sees the true W or b.
# We only have input data and the corresponding target labels.
# The model must learn the underlying W and b by comparing its
# predictions to the ground truth and adjusting its parameters
# to minimize the error.

# Create target labels + noise
y_true = X @ true_W + true_b + torch.randn(N, D_out) * 0.1

# Input and ground truth label should be same size
print("X shape:", X.shape)
print("y_true shape:", y_true.shape)

X shape: torch.Size([10, 1])
y_true shape: torch.Size([10, 1])


 ---
## Initialize the model's parameters

Remember: to make PyTorch track a tensor as a learnable parameter, we must wrap it in torch.nn.Parameter. Otherwise it's treated as a normal tensor and will NOT be updated during backpropagation.
(See lecture 02_autograd if you need a refresher!)

In [47]:
# Initialize the model's parameters with random values.
# W has shape (D_in, D_out) so it can be multiplied 
# with X of shape (N, D_in). 
# b has shape (1,) and will be broadcast across all samples.
# Setting requires_grad=True makes PyTorch track these 
# tensors as learnable parameters whose gradients will be 
# computed during backprop.
W = torch.randn(D_in, D_out, requires_grad=True)
b = torch.randn(1, requires_grad=True)

# What you see is the models initial prediction which
# of course will be completly wrong! 
print(f"Initial W: {W}\n")
print(f"Initial b: {b}")


Initial W: tensor([[-0.5551]], requires_grad=True)

Initial b: tensor([-0.2939], requires_grad=True)


 ---
## The actual implementation

A linear model has a very simple forward pass:

- y_hat = X @ W + b

This is exactly the same operation we used to generate the synthetic data.
Nothing more happens here, this is the entire forward pass.

**Note:**  
In the code below we define a class for our model.  
You **do not** need to fully understand how this works yet, but the code is heavily commented so you can review it later at your own pace.

A PyTorch model class is built by inheriting from  
**torch.nn.Module**, which is the base class for all neural‑network components.  
Inside the class we create a **linear layer** that automatically manages its own weights and biases.  
The forward() method defines how the model computes its predictions during the  
**forward pass**.

For now, just read the comments and continue, you will understand this structure naturally as we progress. The important thing to notice here is that the model’s prediction **y_hat** looks nothing like the true labels
**y_true** at this stage.


In [48]:
# ----- 3. FORWARD PASS -----
# Our manual linear model: y_hat = X @ W + b
y_hat = X @ W + b

# Print the first 5 predicted values.
# These will be random at first because the model has not been trained yet.
# NOTICE THAT grad_fn IS ACTIVE!! It will say SliceBackward
# since our last operation to y_hat is indexing the five first rows
print(f"Prediction y_hat (only first 5 rows):\n{y_hat[:5]}\n")

# Print the first 5 true labels so we can compare.
print(f"True labels y_true (only first 5 rows): \n{y_true[:5]}")


Prediction y_hat (only first 5 rows):
tensor([[-1.0824],
        [ 0.1622],
        [-0.9612],
        [ 0.3032],
        [-1.6149]], grad_fn=<SliceBackward0>)

True labels y_true (only first 5 rows): 
tensor([[ 3.9033],
        [-0.6884],
        [ 3.4467],
        [-1.2623],
        [ 5.8979]])


 ---
## Backward pass
Okay, so our model has made its first prediction, and it was completely wrong. That’s expected at this stage. The model now needs to adjust its internal parameters **W** and **b** so that its next prediction gets closer to the true labels **y_true**.

You can think of W and b as two tunable radio knobs, one for volume and one for frequency. Imagine that the target label is a football game being broadcast at a specific frequency. The model turns the knobs slightly and listens to the output after each tiny adjustment. At first, all it hears is static, because the knobs are set randomly. But with each adjustment, the sound becomes a little clearer.

If the path from turning the knob to hearing the sound is the *forward pass*, then the moment the model realizes “this doesn’t sound right” and figures out how to adjust the knobs again is the *backward pass*.

### Loss
The loss is simply a number that tells the model *how wrong* its prediction was. If y_hat is far away from y_true, the loss will be large. If the prediction is close, the loss will be small.

You can think of the loss as the static you hear on the radio. The more static, the worse the signal, and the more the knobs (W and b) need to be adjusted. As the model tunes the knobs correctly, the static decreases,
and the loss goes down. A perfectly tuned signal would mean a loss of zero.

In other words, the backward pass is where PyTorch computes how much each parameter contributed to the error. These gradients tell the model which direction to move **W** and **b** in order to reduce the loss on the next forward pass.

In our case, the loss function, which is the operation that measures how wrong the model’s prediction is, is computed using Mean Squared Error (MSE). MSE looks at the difference between each predicted value and its true label, squares those differences, and then averages them: 

$$
\text{Loss} = \frac{1}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i)^2
$$

In other words, for every prediction, find the difference (y_hat-y), square that difference to make it positive (y_hat-y)² and take the average of all those differences.

In [49]:
# Calculate error
error = y_hat - y_true
# Square error
squared_error = error ** 2
# Take the average
loss = squared_error.mean()

print(f"Loss: {loss}")

Loss: 15.096049308776855


 ---
## Summary

Our calculated loss does not mean anything on its own, however, this quantification of the wrongness in our model’s prediction is stored at the very end of the computation graph, meaning the loss tensor also has a
grad_fn. This allows the model to trace back through the operations that produced the prediction and understand how each step contributed to the final error. This is the superpower of a neural network: it can compare its output to the true labels, see how wrong it was, and adjust **W** and **b** during each iteration in order to reduce the loss as much as possible.

**Note:**  
The whole idea of a neural network is to decrease the loss as much as possible until the training eventually stagnates, at that point, the training (the tuning of **W** and **b**) is finalized.

 ---
 ---
# The backward pass

We have learned how the model feeds input data through its layers and produces a prediction. We also learned that the quantified error of that prediction is called the loss, and that this loss is the
single most important signal the model uses to learn how to make better predictions by reducing it over iterations. But how do we know which way to turn the W and b knobs to lower the loss?

AUTOGRAD does this for us, with one single command:

loss.backward()

This line of code is the building stone of neural networks, and it is one of the most powerful lines of code in all of PyTorch, maybe even in the history of Python, considering the explosion of AI in the last few years. This single line is responsible for calculating the gradients that allow the model to update its parameters, and it is ultimately behind almost every correct answer you receive from a modern LLM.

What this line of code basically says is 

- "Go backwards from the loss and calculate gradients for all parameters with requires_grad=True"

For the examples in this crash course, it is two very important gradients that will be calculated 

- $ \frac{\partial L}{\partial W} $ The gradient of the Loss with respect to our Weight
- $ \frac{\partial L}{\partial b} $ The gradient of the Loss with respect to our Bias

 ---
## Gradients of MSE with respect to W and b

We use Mean Squared Error (MSE) as our loss function:

$ L = \frac{1}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i)^2 $

where

- $\hat{y}_i$ is the model’s prediction for sample $i$
- $y_i$ is the true label for sample $i$
- $n$ is the number of samples

For a simple linear model, each prediction is:

$ \hat{y}_i = W x_i + b $

where

- $W$ is the weight (our “slope knob”)
- $b$ is the bias (our “offset knob”)
- $x_i$ is the input for sample $i$

---

## Gradient of the loss with respect to W

The gradient of the loss with respect to the weight $W$ is:

$ \frac{\partial L}{\partial W} = \frac{2}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i)\, x_i $

**Intuition:**  
- $(\hat{y}_i - y_i)$ is the error for sample $i$  
- $x_i$ is how strongly that sample “touches” $W$  
- If the error is large and $x_i$ is large, that sample pushes hard on $W$  
- The sign tells us the direction:
  - If the model predicts too high on average → gradient is positive → we need to decrease $W$
  - If the model predicts too low on average → gradient is negative → we need to increase $W$

---

## Gradient of the loss with respect to b

The gradient of the loss with respect to the bias $b$ is:

$ \frac{\partial L}{\partial b} = \frac{2}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i) $

**Intuition:**  
- $b$ is just a constant offset, it does not depend on $x_i$  
- Therefore, each sample contributes only its error $(\hat{y}_i - y_i)$  
- If the model is, on average, predicting too high → the sum of errors is positive → we decrease $b$  
- If the model is, on average, predicting too low → the sum of errors is negative → we increase $b$

---

## Why these gradients matter

These two gradients,

$ \frac{\partial L}{\partial W} \quad \text{and} \quad \frac{\partial L}{\partial b} $

tell us **how to turn the knobs** $W$ and $b$ to reduce the loss:

- We move $W$ and $b$ a small step **in the opposite direction** of their gradients  
- This is what happens under the hood when we call:

loss.backward()


 ---
## loss.backwards()

Okey, so thats the teory. But what happens in PyTorch when we run this line of code? 

- The .grad attribute is populated for our W and b tensors

The .grad attribute holds the result from the gradient calculation and tells us how W and b should be adjusted in order to decrease the loss. 

In [50]:
# Compute the gradients
loss.backward()

print(f"Gradient for W: \n {W.grad}")
print(f"\nGradient for b: {b.grad}")

Gradient for W: 
 tensor([[-8.8234]])

Gradient for b: tensor([-5.6537])


**Notice** the sign of each printed gradient. This is crucial. As mentioned earlier, **a negative gradient means that increasing the value of W or b will decrease the loss**. The gradient always points in the direction of
steepest *increase* in loss, so to minimize the loss we must move in the opposite direction.

With this, we now have two essential pieces of information:

1. We can quantify how wrong the model is (the loss).
2. Through the .grad attribute, we know **which direction to turn the knobs W and b** to reduce that loss.

This process happens every time the model makes a prediction during training. It repeats over and over again, for every batch and every epoch, until the loss stops decreasing and the model no longer improves.

 ---
 ---
You’ve now reached the point where the core ideas behind deep learning are no longer abstract. You understand how data flows **forward** through a model, how errors flow **backward**, and how gradients tell us exactly how to adjust the model’s parameters. That means you’ve mastered the essential math and mechanics that every neural network relies on.

Before moving on, take a moment to recap what you’ve learned. If any part still feels unclear, revisit those sections or look up additional explanations online. These fundamentals are not optional, **they are the foundation for everything that comes next.**

In the next lecture, we’ll turn these concepts into action by building a real training loop that updates W and b step by step. For now, rest, review, and make sure the ideas feel solid. The upcoming material will build directly on this knowledge.
